# 🎚 Steering test · Is `L11/f61723` causal for hallucination calibration?

Follow-up to v0.0.2. We found that `L11/f61723` predicts known vs unknown entities with AUROC 0.8379. **That's correlation. Now we test causality.**

**Design** (Anthropic biology-of-LLM style, single-feature clamp):
1. For ~20 known + ~20 unknown Wikidata entities (re-labelled in this notebook with same v0.0.2 protocol)
2. Generate ~80 tokens of free-form response to `"Tell me about {entity}"` under three conditions:
   - **baseline** (no intervention)
   - **clamp f61723 → 0** ("treat as if model knows") — should ↑ confabulation on unknowns
   - **clamp f61723 → 5.0** ("treat as if model doesn't know") — should ↑ refusal on knowns
3. Score each generation by refusal-pattern regex (same as v0.0.2 labelling)
4. Statistical test: does intervention significantly shift refusal rate?

**Verdict**:
- **Causal**: refusal rate moves > 15pp in expected direction → feature controls behavior, not just predicts it
- **Non-causal**: behavior unchanged → feature reads state but doesn't influence it (epiphenomenal)
- **Mixed**: small effect (5-15pp) → partial causality, narrative depends on direction

**Cost**: ~$5 GPU + ~30 min Colab.

**If passes**: the feature graduates from "correlation" to "controllable" — opens RL reward shaping, steering API, paper-grade mechanistic claim.
**If fails**: honest negative — feature is read-only sensor, not knob. Still useful as detector but no causal claim.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub matplotlib tqdm requests scipy
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 1. Config + load model + L11 SAE

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
STEER_LAYER   = 11
STEER_FEATURE = 61723
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

# Steering values
ABLATE_VALUE   = 0.0     # clamp to 0 (treat as known)
AMPLIFY_VALUE  = 5.0     # clamp to 5.0 (above max natural ~3.2 — "strongly unknown")

# Eval budget
N_KNOWN_EVAL   = 20
N_UNKNOWN_EVAL = 20
MAX_GEN_TOKENS = 80
N_ATTRIBUTES_TO_LABEL = 3
PER_TYPE_CANDIDATES = 80   # over-sample to get enough labelled

import os, math, json, time, random, re
import numpy as np
import requests
from collections import Counter
random.seed(0); torch.manual_seed(0); np.random.seed(0)

from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec

sae_path = hf_hub_download(HF_SAE_REPO, f'sae_L{STEER_LAYER}_latest.safetensors')
sae = TopKSAE(load_file(sae_path), K).to(device).eval()
layer_mod = model.model.language_model.layers[STEER_LAYER]
print(f'  ✓ SAE L{STEER_LAYER} loaded · target feature: f{STEER_FEATURE}')
print(f'  ablate={ABLATE_VALUE} · amplify={AMPLIFY_VALUE}')
print(f'  vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Re-label a small eval set (matches v0.0.2 protocol)

We need entities the model genuinely knows vs doesn't know. Re-running the same labelling on a small over-sample, then taking the first 20 of each.

In [ ]:
from tqdm.auto import tqdm

# Pull Ferrando entity files
FERRANDO_BASE = 'https://raw.githubusercontent.com/javiferran/sae_entities/main/dataset/processed/entities'
ENTITY_TYPES = ['player', 'movie']   # focus on the two types that worked in v0.0.2
raw = {t: requests.get(f'{FERRANDO_BASE}/{t}.json', timeout=60).json() for t in ENTITY_TYPES}
for t, lst in raw.items():
    print(f'  {t}: {len(lst)} candidates available')

rng = random.Random(0)
candidates = []
for t in ENTITY_TYPES:
    sample = rng.sample(raw[t], min(PER_TYPE_CANDIDATES, len(raw[t])))
    for ent in sample:
        candidates.append({'type': t, **ent})
print(f'\ntotal candidates to label: {len(candidates)}')

ATTRIBUTE_TEMPLATES = {
    'player': {
        'place_birth':  "What is the place of birth of the basketball player '{entity}'? Answer in just one or two words.",
        'date_birth':   "In what year was the basketball player '{entity}' born? Answer with just a year.",
        'teams_list':   "What was a team that the basketball player '{entity}' played for? Answer with just the team name.",
    },
    'movie': {
        'directors':    "Who directed the movie '{entity}'? Answer with just the director's name.",
        'release_year': "In what year was the movie '{entity}' released? Answer with just a year.",
        'genres':       "What is one genre of the movie '{entity}'? Answer with just one word.",
    },
}

REFUSAL_PATTERNS = [
    r"i (?:don'?t|do not) (?:know|have)",
    r"i'?m (?:sorry|not sure|not familiar|unable)",
    r"i (?:cannot|can'?t) (?:provide|verify|confirm|find)",
    r"there (?:is|seems to be) (?:no|insufficient|limited) (?:information|data|record)",
    r"unable to (?:find|locate|verify)",
    r"(?:no|not enough|insufficient) (?:public(?:ly available)?\s+)?(?:information|data|record)",
    r"i don't have (?:specific|enough|reliable) (?:information|details|data)",
    r"there'?s no (?:widely|publicly|reliable) (?:known|available)",
]
REFUSAL_RE = re.compile('|'.join(REFUSAL_PATTERNS), re.IGNORECASE)
def is_refusal(text): return bool(REFUSAL_RE.search(text or ''))

def normalise(s): return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()
def attr_match(answer, gt):
    if isinstance(gt, list): return any(attr_match(answer, x) for x in gt)
    if not gt or not answer: return False
    a = normalise(str(answer)); g = normalise(str(gt))
    if not g: return False
    return g in a or a in g

def chat_short(question):
    messages = [
        {'role': 'system', 'content': 'Answer concisely and directly. Do not show reasoning. If you do not know, say "I do not know".'},
        {'role': 'user', 'content': question},
    ]
    try:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids['input_ids'], attention_mask=ids['attention_mask'],
                              max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    if 'thinking process' in ans.lower():
        parts = [p for p in ans.split('\n') if p.strip()]
        ans = parts[-1] if parts else ans
    return ans

labelled = []
for ent in tqdm(candidates, desc='labelling'):
    name = ent['entity']; type_ = ent['type']
    gt = {}
    for a in ent.get('attributes', []):
        gt.setdefault(a['attribute_type'], []).append(a['attribute_value'])
    templates = ATTRIBUTE_TEMPLATES[type_]
    avail = [k for k in templates if k in gt]
    if len(avail) < 2: continue
    selected = avail[:N_ATTRIBUTES_TO_LABEL]
    n_correct = n_refuse = 0
    for attr in selected:
        a = chat_short(templates[attr].format(entity=name))
        if is_refusal(a): n_refuse += 1
        elif attr_match(a, gt[attr]): n_correct += 1
    if n_correct >= 2 and n_refuse == 0: cls = 'known'
    elif n_correct == 0 and n_refuse >= 1: cls = 'unknown'
    else: cls = 'middle'
    labelled.append({'type': type_, 'entity': name, 'class': cls})
    if cls != 'middle' and Counter(l['class'] for l in labelled).get(cls, 0) >= max(N_KNOWN_EVAL, N_UNKNOWN_EVAL) * 1.5:
        # have plenty — keep going to balance, but stop when both classes are ≥ goal
        c = Counter(l['class'] for l in labelled)
        if c.get('known', 0) >= N_KNOWN_EVAL and c.get('unknown', 0) >= N_UNKNOWN_EVAL:
            break

known_eval   = [l for l in labelled if l['class'] == 'known'][:N_KNOWN_EVAL]
unknown_eval = [l for l in labelled if l['class'] == 'unknown'][:N_UNKNOWN_EVAL]
print(f'\neval set: {len(known_eval)} known + {len(unknown_eval)} unknown')
print(f'  known sample: {[e["entity"] for e in known_eval[:5]]}')
print(f'  unknown sample: {[e["entity"] for e in unknown_eval[:5]]}')

## 3. Steering hook — clamp `f{STEER_FEATURE}` at L11 to a fixed value

On every forward pass through L11 (during prompt processing AND token-by-token generation), we:
1. Encode the residual through the SAE
2. Clamp the target feature to `clamp_value` (overrides natural activation)
3. Decode back, re-add the SAE error term so other features are unaffected

Equivalent to: `residual = sae.decode(z_with_clamped_feat) + (residual - sae.decode(z_original))`.

In [ ]:
_active_clamp = {'value': None}   # None = baseline (no intervention)

def steer_hook(mod, inp, out):
    if _active_clamp['value'] is None:
        return out
    clamp = _active_clamp['value']
    h = out[0] if isinstance(out, tuple) else out
    orig_dtype = h.dtype
    flat = h.reshape(-1, D_MODEL).to(torch.bfloat16)
    z = sae.encode(flat)
    recon_orig = sae.decode(z).to(torch.float32)
    err = flat.to(torch.float32) - recon_orig
    z_mod = z.clone()
    z_mod[:, STEER_FEATURE] = clamp
    recon_new = sae.decode(z_mod).to(torch.float32)
    new_h = (recon_new + err).to(orig_dtype).reshape(h.shape)
    if isinstance(out, tuple):
        return (new_h,) + out[1:]
    return new_h

_steer_handle = layer_mod.register_forward_hook(steer_hook)   # always on, controlled by _active_clamp

def generate_under(condition, entity_name):
    """condition: 'baseline' / 'ablate' / 'amplify'."""
    if condition == 'baseline':
        _active_clamp['value'] = None
    elif condition == 'ablate':
        _active_clamp['value'] = ABLATE_VALUE
    elif condition == 'amplify':
        _active_clamp['value'] = AMPLIFY_VALUE
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant. Answer briefly with concrete facts. If you do not know, say "I do not know".'},
        {'role': 'user', 'content': f"Tell me three concrete facts about '{entity_name}'."},
    ]
    try:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            ids['input_ids'], attention_mask=ids['attention_mask'],
            max_new_tokens=MAX_GEN_TOKENS, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    _active_clamp['value'] = None
    response = tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return response

# Smoke test on 1 known + 1 unknown × 3 conditions
print('Smoke test:')
for label, ent in [('known', known_eval[0]), ('unknown', unknown_eval[0])]:
    print(f"\n[{label}] {ent['entity']}")
    for cond in ['baseline', 'ablate', 'amplify']:
        r = generate_under(cond, ent['entity'])
        print(f'  [{cond:>9s}] {r[:150]}')

## 4. Run full steering experiment — 40 entities × 3 conditions = 120 generations

In [ ]:
results = []
all_eval = [(e, 'known') for e in known_eval] + [(e, 'unknown') for e in unknown_eval]

for ent, ent_class in tqdm(all_eval, desc='steering'):
    name = ent['entity']
    row = {'entity': name, 'type': ent['type'], 'class': ent_class}
    for cond in ['baseline', 'ablate', 'amplify']:
        r = generate_under(cond, name)
        row[f'{cond}_text']    = r
        row[f'{cond}_refusal'] = is_refusal(r)
    results.append(row)

print(f'\ngenerated {len(results) * 3} responses')
print(f'sample baseline:  {results[0]["baseline_text"][:120]}')
print(f'sample ablate:    {results[0]["ablate_text"][:120]}')
print(f'sample amplify:   {results[0]["amplify_text"][:120]}')

## 5. Effect sizes + statistical test

In [ ]:
from scipy.stats import binomtest

def rate(rows, cond):
    return sum(r[f'{cond}_refusal'] for r in rows) / max(len(rows), 1)

known_rows   = [r for r in results if r['class'] == 'known']
unknown_rows = [r for r in results if r['class'] == 'unknown']

print('═' * 60)
print('Refusal rate by class × intervention')
print('═' * 60)
print(f'                     baseline    ablate(→0)   amplify(→{AMPLIFY_VALUE})')
for label, rows in [('KNOWN entities  ', known_rows), ('UNKNOWN entities', unknown_rows)]:
    rb = rate(rows, 'baseline')
    ra = rate(rows, 'ablate')
    rm = rate(rows, 'amplify')
    print(f'  {label}    {rb:>5.1%}      {ra:>5.1%}        {rm:>5.1%}    (n={len(rows)})')

# Effect sizes
delta_amp_on_known = rate(known_rows, 'amplify') - rate(known_rows, 'baseline')
delta_abl_on_unk   = rate(unknown_rows, 'ablate') - rate(unknown_rows, 'baseline')

print('\nKey effect sizes:')
print(f'  Δ(refusal | amplify, known)   = {delta_amp_on_known:+.1%}    expected: ↑ (positive)')
print(f'  Δ(refusal | ablate, unknown)  = {delta_abl_on_unk:+.1%}    expected: ↓ (negative)')

# Sign test: how many entities flipped in expected direction?
amp_flips = sum(1 for r in known_rows if r['amplify_refusal'] and not r['baseline_refusal'])
abl_unflips = sum(1 for r in unknown_rows if not r['ablate_refusal'] and r['baseline_refusal'])

print(f'\nPer-entity flips:')
print(f'  amplify on known: {amp_flips}/{len(known_rows)} flipped baseline-NO-refuse → amplify-DO-refuse')
print(f'  ablate on unknown: {abl_unflips}/{len(unknown_rows)} flipped baseline-DO-refuse → ablate-NO-refuse')

# Verdict
delta_score = max(abs(delta_amp_on_known), abs(delta_abl_on_unk))
if delta_score >= 0.15:
    verdict = 'CAUSAL · feature controls behavior'
    flag = '\u2705'
elif delta_score >= 0.05:
    verdict = 'PARTIALLY CAUSAL · small but real effect'
    flag = '\u26a0\ufe0f'
else:
    verdict = 'NOT CAUSAL · feature reads state but does not control it'
    flag = '\u274c'

print(f'\n{flag}  VERDICT: {verdict}  (max |Δ| = {delta_score:.1%})')

## 6. Qualitative inspection — show before/after on 3 examples per direction

In [ ]:
# Show 3 known entities where amplify changed refusal status
print('═' * 60)
print('AMPLIFY (→{}) on KNOWN entities — should make refusals happen'.format(AMPLIFY_VALUE))
print('═' * 60)
shown = 0
for r in known_rows:
    if shown >= 3: break
    if r['amplify_refusal'] != r['baseline_refusal']:
        print(f"\n● {r['entity']} ({r['type']})")
        print(f'  baseline (no refusal): {r["baseline_text"][:200]}')
        print(f'  amplify  (refusal={r["amplify_refusal"]}): {r["amplify_text"][:200]}')
        shown += 1
if shown == 0:
    print('  (no entities flipped from no-refusal to refusal under amplify)')

print('\n' + '═' * 60)
print('ABLATE (→0) on UNKNOWN entities — should suppress refusals')
print('═' * 60)
shown = 0
for r in unknown_rows:
    if shown >= 3: break
    if r['ablate_refusal'] != r['baseline_refusal']:
        print(f"\n● {r['entity']} ({r['type']})")
        print(f'  baseline (refusal={r["baseline_refusal"]}): {r["baseline_text"][:200]}')
        print(f'  ablate   (refusal={r["ablate_refusal"]}): {r["ablate_text"][:200]}')
        shown += 1
if shown == 0:
    print('  (no entities flipped under ablate)')

## 7. Save artifacts to HF

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

out = {
    'version':       'v0.0.1',
    'experiment':    f'steering_L{STEER_LAYER}_f{STEER_FEATURE}',
    'method':        'single-feature SAE clamp during generation, refusal-rate scoring',
    'model':         HF_BASE_MODEL,
    'sae_repo':      HF_SAE_REPO,
    'feature':       {'layer': STEER_LAYER, 'feature_id': STEER_FEATURE},
    'clamp_values':  {'baseline': None, 'ablate': ABLATE_VALUE, 'amplify': AMPLIFY_VALUE},
    'n_known':       len(known_rows), 'n_unknown': len(unknown_rows),
    'rates': {
        'known':   {c: rate(known_rows, c) for c in ['baseline', 'ablate', 'amplify']},
        'unknown': {c: rate(unknown_rows, c) for c in ['baseline', 'ablate', 'amplify']},
    },
    'effect_sizes': {
        'amplify_on_known_delta':  delta_amp_on_known,
        'ablate_on_unknown_delta': delta_abl_on_unk,
    },
    'verdict':       verdict,
    'predecessor':   'hallucination_v0_0_2.json (correlational AUROC=0.8379)',
    'samples_per_entity': [
        {k: v for k, v in r.items() if k != 'class' or k != 'type'} for r in results[:6]
    ],
    'timestamp':     datetime.now(timezone.utc).isoformat(),
}
with open('/tmp/steering_v001.json', 'w') as f:
    json.dump(out, f, indent=2)

api = HfApi()
api.upload_file(
    path_or_fileobj='/tmp/steering_v001.json',
    path_in_repo='steering_v0_0_1.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Steering v0.0.1 — {verdict.split(chr(183))[0].strip()} (max |Δ|={delta_score:.1%})',
)
print(f'\n✓ uploaded → https://huggingface.co/{HF_SAE_REPO}/blob/main/steering_v0_0_1.json')
print(f'verdict: {verdict}')